### Activity 1: Review the cleaned dataset 
 Load the cleaned dataset from data/processed/ted_talks_cleaned.csv. 

 Inspect the available columns. 

 Confirm that the title and combined text fields are available. 

 Check the number of usable records. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

In [3]:
df.head()

,Unnamed: 0,talk_id,title,speaker_1,duration,topics,url,description,transcript,title_clean,speaker_1_clean,topics_clean,description_clean,transcript_clean,combined_text
0,0,1,Averting the climate crisis,Al Gore,977,"['alternative energy', 'cars', 'climate change...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",averting the climate crisis,al gore,alternative energy cars climate change culture...,with the same humor and humanity he exuded in ...,"thank you so much, chris. and it's truly a gre...",averting the climate crisis alternative energy...
1,1,92,The best stats you've ever seen,Hans Rosling,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...",the best stats you've ever seen,hans rosling,africa asia google demo economics global issue...,you've never seen data presented like this. wi...,"about 10 years ago, i took on the task to teac...",the best stats you've ever seen africa asia go...
2,2,7,Simplicity sells,David Pogue,1286,"['computers', 'entertainment', 'interface desi...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",simplicity sells,david pogue,computers entertainment interface design media...,new york times columnist david pogue takes aim...,"(music: ""the sound of silence,"" simon & garfun...",simplicity sells computers entertainment inter...
3,3,53,Greening the ghetto,Majora Carter,1116,"['MacArthur grant', 'activism', 'business', 'c...",https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today â€” and I'm very happy th...,greening the ghetto,majora carter,macarthur grant activism business cities envir...,"in an emotionally charged talk, macarthur-winn...",if you're here today â€” and i'm very happy th...,greening the ghetto macarthur grant activism b...
4,4,66,Do schools kill creativity?,Sir Ken Robinson,1164,"['children', 'creativity', 'culture', 'dance',...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,do schools kill creativity?,sir ken robinson,children creativity culture dance education pa...,sir ken robinson makes an entertaining and pro...,good morning. how are you? (audience) good. it...,do schools kill creativity? children creativit...


In [4]:
df.columns.tolist()

['Unnamed: 0',
 'talk_id',
 'title',
 'speaker_1',
 'duration',
 'topics',
 'url',
 'description',
 'transcript',
 'title_clean',
 'speaker_1_clean',
 'topics_clean',
 'description_clean',
 'transcript_clean',
 'combined_text']

In [5]:
df = df.drop(columns=['Unnamed: 0'])

In [6]:
df.shape

(4005, 14)

### Activity 2: Implement TF-IDF vectorization 
 Use the TfidfVectorizer class from scikit-learn. 

 Convert the combined text field into a TF-IDF matrix. 

 Inspect the vocabulary and matrix dimensions. 

 Explain what the rows, columns, and numerical values represent. 

In [7]:
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize

In [8]:
stop_words = set(stopwords.words("english"))
stemmer = SnowballStemmer("english")

In [9]:
def preprocess_text(text):
    if pd.isna(text):
        return ""

    words = word_tokenize(str(text).lower())

    processed_words = [
        stemmer.stem(word)
        for word in words
        if word.isalpha() and word not in stop_words
    ]

    return " ".join(processed_words)


df["processed_text"] = df["combined_text"].apply(preprocess_text)

# Inspect results
display(df[["combined_text", "processed_text"]].head())

,combined_text,processed_text
0,averting the climate crisis alternative energy...,avert climat crisi altern energi car climat ch...
1,the best stats you've ever seen africa asia go...,best stat ever seen africa asia googl demo eco...
2,simplicity sells computers entertainment inter...,simplic sell comput entertain interfac design ...
3,greening the ghetto macarthur grant activism b...,green ghetto macarthur grant activ busi citi e...
4,do schools kill creativity? children creativit...,school kill creativ children creativ cultur da...


In [10]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000 # max_features=10000 limits the vocabulary to at most 10,000 terms. The actual number may be lower.
)

X = tfidf_vectorizer.fit_transform(df["processed_text"])

print("TF-IDF matrix shape:", X.shape)
print("Number of talks:", X.shape[0])
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF matrix shape: (4005, 10000)
Number of talks: 4005
Vocabulary size: 10000


### Activity 3: Perform similarity analysis 

 Use cosine similarity to compare talk representations. 

 Examine similarity scores for selected talks. 

 Identify which talks have the highest similarity scores. 

 Explain why a talk should not recommend itself. 

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between every pair of TED Talks
similarity_matrix = cosine_similarity(X)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (4005, 4005)


In [12]:
similarity_matrix

array([[1.        , 0.16826042, 0.24666323, ..., 0.06683369, 0.08676228,
        0.02514766],
       [0.16826042, 1.        , 0.17747084, ..., 0.05472102, 0.05920698,
        0.02476043],
       [0.24666323, 0.17747084, 1.        , ..., 0.05208313, 0.08504583,
        0.02483419],
       ...,
       [0.06683369, 0.05472102, 0.05208313, ..., 1.        , 0.03888523,
        0.00711247],
       [0.08676228, 0.05920698, 0.08504583, ..., 0.03888523, 1.        ,
        0.00736855],
       [0.02514766, 0.02476043, 0.02483419, ..., 0.00711247, 0.00736855,
        1.        ]], shape=(4005, 4005))

In [13]:
similarity_matrix[0]

array([1.        , 0.16826042, 0.24666323, ..., 0.06683369, 0.08676228,
       0.02514766], shape=(4005,))

In [14]:
# Similarity scores of the first talk against all talks
scores = similarity_matrix[0]

print("Talk title:", df.iloc[0]["title"])
print("Similarity with itself:", scores[0])
print("Highest similarity score:", scores.max())

Talk title: Averting the climate crisis
Similarity with itself: 1.0
Highest similarity score: 1.0


In [15]:
# Select the first TED Talk
talk_index = 0

# Get similarity scores for this talk
scores = similarity_matrix[talk_index].copy()

# Prevent the talk from recommending itself
scores[talk_index] = -1

# Get indices of the 5 most similar talks
top_indices = np.argsort(scores)[::-1][:5]

# Display the recommendations
recommendations = df.iloc[top_indices][["talk_id", "title", "speaker_1"]].copy()
recommendations["similarity_score"] = scores[top_indices]

display(recommendations)

,talk_id,title,speaker_1,similarity_score
209,243,New thinking on the climate crisis,Al Gore,0.349718
3134,32560,The disarming case to act right now on climate...,Greta Thunberg,0.321973
363,436,Design and discovery,David Carson,0.318803
2092,2441,The case for optimism on climate change,Al Gore,0.314783
2006,2339,Climate change is happening. Here's how we adapt,Alice Bows-Larkin,0.313717


In [16]:
def recommend_similar_talks(talk_index, top_n=5):
    scores = similarity_matrix[talk_index].copy()

    # Exclude the selected talk itself
    scores[talk_index] = -1

    top_indices = np.argsort(scores)[::-1][:top_n]

    result = df.iloc[top_indices][
        ["talk_id", "title", "speaker_1"]
    ].copy()

    result["similarity_score"] = scores[top_indices]

    print("Selected talk:", df.iloc[talk_index]["title"])
    display(result)

In [17]:
recommend_similar_talks(10, top_n=5)

Selected talk: My wish: Help me stop pandemics


,talk_id,title,speaker_1,similarity_score
158,177,The case for optimism,Larry Brilliant,0.515178
3999,62748,A global pandemic calls for global solutions,Larry Brilliant,0.452970
901,1153,How we'll stop polio for good,Bruce Aylward,0.341411
430,529,Lessons from the 1918 flu,Laurie Garrett,0.328389
663,869,HIV and flu -- the vaccine strategy,Seth Berkley,0.246764


### Why should a talk not recommend itself?

Cosine similarity measures how closely two vectors point in the same direction. A talk compared with its own TF-IDF vector has a similarity score of 1.0, indicating a perfect match.

However, recommending the same talk the user is already viewing is not useful. Therefore, we exclude its own row index before sorting the remaining scores.

scores[talk_index] = -1

### Activity 4: Generate sample recommendations 

 Select at least three different TED Talks. 

 Generate the top five most similar talks for each selected talk, where enough candidate 
records are available. 

 Record the selected talk, recommended talk titles, and similarity scores. 

 Manually review whether the recommendations appear relevant. 

In [18]:
selected_indices = [0, 100, 500]

display(
    df.iloc[selected_indices][
        ["talk_id", "title", "speaker_1"]
    ].reset_index(drop=True)
)

,talk_id,title,speaker_1
0,1,Averting the climate crisis,Al Gore
1,119,"""Black Men Ski""",Stew
2,629,How I swam the North Pole,Lewis Pugh


In [19]:
selected_indices = [0, 100, 500]

recommendation_records = []

for talk_index in selected_indices:

    # Get the selected talk's information
    selected_talk = df.iloc[talk_index]

    # Get its similarity scores against all talks
    scores = similarity_matrix[talk_index].copy()

    # Exclude the selected talk itself
    scores[talk_index] = -1

    # Get the indices of the top 5 similar talks
    top_indices = np.argsort(scores)[::-1][:5]

    # Store each recommendation
    for rank, recommended_index in enumerate(top_indices, start=1):
        recommended_talk = df.iloc[recommended_index]

        recommendation_records.append({
            "selected_talk": selected_talk["title"],
            "selected_speaker": selected_talk["speaker_1"],
            "rank": rank,
            "recommended_talk": recommended_talk["title"],
            "recommended_speaker": recommended_talk["speaker_1"],
            "similarity_score": scores[recommended_index]
        })

# Create the final recommendations table
recommendations_df = pd.DataFrame(recommendation_records)

display(recommendations_df.round({"similarity_score": 4}))

,selected_talk,selected_speaker,rank,recommended_talk,recommended_speaker,similarity_score
0,Averting the climate crisis,Al Gore,1,New thinking on the climate crisis,Al Gore,0.3497
1,Averting the climate crisis,Al Gore,2,The disarming case to act right now on climate...,Greta Thunberg,0.3220
2,Averting the climate crisis,Al Gore,3,Design and discovery,David Carson,0.3188
3,Averting the climate crisis,Al Gore,4,The case for optimism on climate change,Al Gore,0.3148
4,Averting the climate crisis,Al Gore,5,Climate change is happening. Here's how we adapt,Alice Bows-Larkin,0.3137
5,"""Black Men Ski""",Stew,1,How to overcome our biases? Walk boldly toward...,VernÄ Myers,0.3780
6,"""Black Men Ski""",Stew,2,Why gender equality is good for everyone â€” m...,Michael Kimmel,0.2689
7,"""Black Men Ski""",Stew,3,Why did I ski to the North Pole?,Ben Saunders,0.2674
8,"""Black Men Ski""",Stew,4,Violence against women -- it's a men's issue,Jackson Katz,0.2545
9,"""Black Men Ski""",Stew,5,How barbershops can keep men healthy,Joseph Ravenell,0.2384


In [20]:
for talk_index in selected_indices:

    selected_title = df.iloc[talk_index]["title"]

    print("\nSelected TED Talk:", selected_title)

    display(
        recommendations_df[
            recommendations_df["selected_talk"] == selected_title
        ][
            [
                "rank",
                "recommended_talk",
                "recommended_speaker",
                "similarity_score"
            ]
        ].reset_index(drop=True).round({"similarity_score": 4})
    )


Selected TED Talk: Averting the climate crisis


,rank,recommended_talk,recommended_speaker,similarity_score
0,1,New thinking on the climate crisis,Al Gore,0.3497
1,2,The disarming case to act right now on climate...,Greta Thunberg,0.3220
2,3,Design and discovery,David Carson,0.3188
3,4,The case for optimism on climate change,Al Gore,0.3148
4,5,Climate change is happening. Here's how we adapt,Alice Bows-Larkin,0.3137



Selected TED Talk: "Black Men Ski"


,rank,recommended_talk,recommended_speaker,similarity_score
0,1,How to overcome our biases? Walk boldly toward...,VernÄ Myers,0.3780
1,2,Why gender equality is good for everyone â€” m...,Michael Kimmel,0.2689
2,3,Why did I ski to the North Pole?,Ben Saunders,0.2674
3,4,Violence against women -- it's a men's issue,Jackson Katz,0.2545
4,5,How barbershops can keep men healthy,Joseph Ravenell,0.2384



Selected TED Talk: How I swam the North Pole


,rank,recommended_talk,recommended_speaker,similarity_score
0,1,My mind-shifting Everest swim,Lewis Pugh,0.4810
1,2,Extreme swimming with the world's most dangero...,Diana Nyad,0.4357
2,3,What open water swimming taught me about resil...,Bhakti Sharma,0.4188
3,4,Why did I ski to the North Pole?,Ben Saunders,0.4154
4,5,Let's save the last pristine continent,Robert Swan,0.3480


### Activity 5: Analyze limitations 


Document at least three observations about your results. For example, you may find that talks 
with similar words are recommended even when their meaning differs, or that related talks using 
different vocabulary receive lower similarity scores. 

Do not claim that a high similarity score guarantees that a user will like a recommendation.

### Observations from results

Observation 1: Similar words do not always indicate similar meaning.

Observation 2: Some recommendations are more relevant than others.

Observation 3: Related topics can be identified through shared vocabulary.

Observation 4: TF-IDF does not fully understand context or semantics.

Observation 5: Similarity scores do not represent user preferences.